<a href="https://colab.research.google.com/github/MrPhipps/Colabs/blob/main/Qwen_Image_2512.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U diffusers

## Local Inference on GPU
Model page: https://huggingface.co/Qwen/Qwen-Image-2512

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/Qwen/Qwen-Image-2512)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [2]:
pip install -U diffusers transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 117.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
import torch
from diffusers import DiffusionPipeline
from IPython.display import display, clear_output
import ipywidgets as widgets
import os

# Initialize pipe globally, but it will be None until loaded
global pipe
pipe = None

# HF Token input field
hf_token_input = widgets.Text(
    value=os.environ.get('HF_TOKEN', ''), # Pre-fill from environment if available
    placeholder='Enter your Hugging Face Token here',
    description='HF Token:',
    disabled=False
)
display(hf_token_input)

# Button to load the model
load_model_button = widgets.Button(description="Load Model")
display(load_model_button)

# Prompt input field (initially disabled)
prompt_input = widgets.Text(
    value='Astronaut in a jungle, cold color palette, muted colors, detailed, 8k',
    placeholder='Type your prompt here',
    description='Prompt:',
    disabled=True # Disabled until model is loaded
)
display(prompt_input)

# Output widget to display images and messages
output_area = widgets.Output()
display(output_area)


def load_model_and_enable_generation(sender):
    global pipe
    with output_area:
        clear_output(wait=True) # Clear previous output in this area
        token = hf_token_input.value
        if token:
            os.environ['HF_TOKEN'] = token
            print("HF_TOKEN set successfully. Attempting to load model...")
        else:
            print("No HF Token provided. Model might load with rate limits if accessing private models or exceeding public rate limits.")

        # switch to "mps" for apple devices
        try:
            pipe = DiffusionPipeline.from_pretrained("Qwen/Qwen-Image-2512", dtype=torch.bfloat16, device_map="cuda")
            print("Model loaded successfully!")
            prompt_input.disabled = False # Enable prompt input
            load_model_button.disabled = True # Disable load button
            generate_image_button.disabled = False # Enable generate button
        except Exception as e:
            print(f"Error loading model: {e}")
            prompt_input.disabled = True
            generate_image_button.disabled = True

def generate_image(sender):
    global pipe
    if pipe is None:
        with output_area:
            print("Model not loaded. Please load the model first.")
        return

    prompt = prompt_input.value
    with output_area:
        clear_output(wait=True) # Clear previous image
        print(f"Generating image for prompt: '{prompt}'...")
        try:
            image = pipe(prompt).images[0]
            display(image)
        except Exception as e:
            print(f"Error generating image: {e}")

# Button to trigger image generation (separate from on_submit for explicit control)
generate_image_button = widgets.Button(description="Generate Image", disabled=True)
display(generate_image_button)

# Attach event handlers
load_model_button.on_click(load_model_and_enable_generation)
generate_image_button.on_click(generate_image)
prompt_input.on_submit(generate_image) # Keep on_submit for convenience

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

Fetching 27 files:   0%|          | 0/27 [00:00<?, ?it/s]

Keyword arguments {'dtype': torch.bfloat16} are not expected by QwenImagePipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 38.05 GiB. GPU 0 has a total capacity of 14.56 GiB of which 14.46 GiB is free. Including non-PyTorch memory, this process has 102.00 MiB memory in use. Of the allocated memory 0 bytes is allocated by PyTorch, and 0 bytes is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Remote Inference via Inference Providers
Ensure you have a valid **HF_TOKEN** set in your environment. You can get your token from [your settings page](https://huggingface.co/settings/tokens). Note: running this may incur charges above the free tier.
The following Python example shows how to run the model remotely on HF Inference Providers, automatically selecting an available inference provider for you.
For more information on how to use the Inference Providers, please refer to our [documentation and guides](https://huggingface.co/docs/inference-providers/en/index).

In [ ]:
import os
os.environ['HF_TOKEN'] = 'YOUR_TOKEN_HERE'

In [ ]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="auto",
    api_key=os.environ["HF_TOKEN"],
)

# output is a PIL.Image object
image = client.text_to_image(
    "Astronaut riding a horse",
    model="Qwen/Qwen-Image-2512",
)